# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadahannadeembaig/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
import os
if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/muhammadahannadeembaig/flyrank-ml-internship.git
%cd flyrank-ml-internship

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.shape

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 124 (delta 39), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 10.44 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/flyrank-ml-internship/flyrank-ml-internship


(30000, 44)

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Content Refresh / Opportunity Scoring. This is a ranking/scoring task, not classification — because the goal isn't to sort pages into a simple yes/no category, but to arrange them in order of priority. The output feeds directly into a content action: the content team will review and refresh the highest-scoring pages first, since they have limited time and need to know where to start.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [14]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

I want to predict trend_pct (and the related trend_direction column). This is a proxy, not a direct measurement, because the outcome I actually care about — how much traffic a page will lose in the future if it isn't refreshed — can't be observed yet since it hasn't happened. Instead, trend_pct captures the recent percentage change in the page's performance, which is already present in the current data and correlates with future decline.

In [15]:
df[['trend_direction', 'trend_pct']].head(10)

,trend_direction,trend_pct
0,down,-41.4
1,down,-57.7
2,down,-60.9
3,stable,-13.8
4,down,-34.7
5,down,-38.9
6,down,-92.3
7,stable,0.6
8,down,-58.8
9,down,-29.2


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The success metric is Precision@K, specifically Precision@50. It measures how many of the top 50 pages the model flags as high priority (most negative trend_pct) are genuinely worth refreshing. This metric fits the real workflow, since the content team can only act on a limited number of pages at a time — what matters is that the top of the list is trustworthy, not the accuracy across the entire dataset.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [16]:
print(f"Total rows: {len(df)}")
df.head(10)

Total rows: 30000


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


Each row represents one page (identified by content_id).

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule, such as "flag any page whose trend_pct dropped below -30%," breaks down quickly in practice. Different pages have different baseline traffic and content types, so a single threshold doesn't apply fairly across all of them. On top of that, priority depends on several factors interacting together — the trend percentage, how outdated the page is (days_since_last_update), and how much visibility it has lost — and no simple if-statement can capture how those combine. A model can learn these patterns directly from the data instead of relying on someone manually guessing the right thresholds, which is why this is genuinely an ML problem rather than something a fixed rule could handle.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.